In [ ]:
# Use %pip so installs go to this notebook kernel's environment
%pip install -q pandas instaquery numpy matplotlib openai snowflake-connector-python


In [1]:
# Import libraries
import pandas as pd
import instaquery as iq
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import json
# Import librare
import instaquery as iq
import datetime
import numpy as np
import os
import logging 
snowflake_logger = logging.getLogger('snowflake.connector')
snowflake_logger.setLevel(logging.WARNING)
# Import regex if not already imported
import re

# Pre-Processing: Summarize contact content to one-sentence

In [2]:
sql_query = """
WITH retail_agents AS (

SELECT
    ca.name
    , ca.id 
    -- , ca.UJET_AGENT_ID
    , ca.email
    , ra.team 
    , ra.supervisor
FROM etl.support_tool_users AS ca
INNER JOIN
    (
    SELECT
        agents.*,
        LOWER(agents.work_email) as email
        , SUBSTRING(agents.BUSINESS_TITLE, 21, 12) as team
        , manager.FIRST_NAME || ' ' ||manager.LAST_NAME as supervisor
        -- , agents.FIRST_NAME  || ' ' ||agents.LAST_NAME as specialist_name
        -- , manager2.FIRST_NAME || ' ' ||manager2.LAST_NAME as supervisor
    FROM dwh.workday_team_hierarchy as agents
    LEFT JOIN dwh.workday_team_hierarchy as manager
        ON agents.manager_id = manager.employee_id
    LEFT JOIN dwh.workday_team_hierarchy as manager2
        ON manager.manager_id = manager2.employee_id
    
    WHERE true
        AND agents.cost_center = 'Customer Experience Support'
        AND agents.BUSINESS_TITLE IN
        (
         'Customer Experience Retail Voice Specialist',
         'Customer Experience Retail Voice Specialists',
         'Customer Experience Retail Email Specialist',
         'Customer Experience Retail Voice Specialist II',
         'Customer Experience Retail Email Specialist II'
        )
    ) AS ra ON ca.email = ra.email

-- WHERE team = 'Retail Email'
)

,email_base as 
(SELECT 
    FSC.primary_contact_id,
    CSE.SUBJECT,
    FSC.subcase_type,
    FSC.CONTACT_CHANNEL,
    CSE.TEXT_BODY,
    CONTACT_CREATED_AT_UTC,
    CSE.CREATED_DATE,
    max(case when ca.id is not null then 1 else 0 end) as is_retail_agent,
    rank() over (partition by primary_contact_id order by CSE.CREATED_DATE asc) as message_rank,
    rank() over (partition by primary_contact_id order by CSE.CREATED_DATE desc) as message_rank_desc

FROM cx_support.salesforce_eclipse.emailmessage CSE    
JOIN INSTADATA.ETL.FACT_SUPPORT_CONTACTS FSC
ON cse.related_to_id = fsc.ticket_id::varchar
AND FSC.contact_channel = 'email'
INNER JOIN etl.fact_support_touches AS fst
ON fst.contact_id = fsc.id
LEFT JOIN retail_agents AS ca
ON ca.id = fst.agent_id
WHERE CONTACT_CREATED_AT_UTC::DATE >= '2025-05-01'
AND CONTACT_CREATED_AT_UTC::DATE < '2025-11-01'
AND FSC.USER_CHANNEL = 'retailer'
AND FSC.TICKET_ID_SOURCE = 'salesforce'
group by 1,2,3,4,5,6,7)

SELECT 
    primary_contact_id,
    CONTACT_CHANNEL,
    min(CREATED_DATE) as transcript_created_date_at_utc,
    max(is_retail_agent) as is_retail_agent,
    MAX(CASE WHEN message_rank = 1 THEN subject END) AS subject,
    LISTAGG(
        'Message ' || message_rank || ' (' || contact_created_at_utc || '):\n' || text_body, 
        '\n\n=== NEXT MESSAGE ===\n\n'
    ) WITHIN GROUP (ORDER BY CREATED_DATE) AS transcript
    --COUNT(*) as message_count
FROM email_base 
GROUP BY 1,2
--where (message_rank = 1 or message_rank_desc = 1)

UNION ALL

SELECT
    FSC.primary_contact_id,
    FSC.CONTACT_CHANNEL,
    min(CT.CREATED_AT_UTC) as transcript_created_date_at_utc,
    max(case when ca.id is not null then 1 else 0 end) as is_retail_agent,
    null as subject,
    max(CT.transcript) as transcript

FROM instadata.etl_eclipse.support_call_transcripts CT
JOIN INSTADATA.ETL.FACT_SUPPORT_CONTACTS FSC
ON FSC.primary_contact_id::varchar = CT.AUDIO_ID::varchar
AND CT.audio_source = replace(split(fsc.primary_contact_id_source, '_')[0],'"','')
INNER JOIN etl.fact_support_touches AS fst
ON fst.contact_id = fsc.id
LEFT JOIN retail_agents AS ca
ON ca.id = fst.agent_id
WHERE CONTACT_CREATED_AT_UTC::DATE >= '2025-05-01'
AND CONTACT_CREATED_AT_UTC::DATE < '2025-11-01'
AND FSC.USER_CHANNEL = 'retailer'
AND FSC.TICKET_ID_SOURCE = 'salesforce'
group by 1,2;
"""

In [3]:
results_df = iq.query(sql_query)

In [4]:
# @title Connecting with OpenAI
from openai import OpenAI

## Enter the name of the source you have created via the website or Bento UI
SOURCE = 'srividyasekar-personal'


client = OpenAI(
    # AI Gateway does not need or want an API key, passed unused because its required here
    api_key="unused",
    base_url=f"https://aigateway.instacart.tools/proxy/{SOURCE}/openai/v1"
)

In [ ]:
# df = pd.read_csv('/Users/ashleyhan/Documents/data/cx_retailer_sampling.csv')
# df.head()

## summarization 

In [5]:
system_prompt = """You are a customer experience analyst specializing in contact reason classification. 
Your task is to read customer service transcripts and create a single, concise sentence that captures 
the PRIMARY reason the audience contacted support.

Guidelines:
- Focus on the customer's CORE ISSUE or REQUEST, not secondary concerns
- Use neutral, descriptive language (avoid subjective terms)
- Capture WHAT happened and WHAT the customer wanted
- Start with an action verb when possible (e.g., "Customer/Shopper/Retailer reported...", "Customer/Shopper/Retailer requested...", "Customer/Shopper/Retailer inquired...")
- Include key details: order/delivery issues, payment problems, product concerns, account issues, etc.
- Keep it under 20 words
- Do NOT include:
  * Agent names or internal processes
  * Pleasantries or conversational filler
  * Resolution details (focus on the initial problem)
  * Multiple issues (choose the primary one)

Examples:
- "Customer reported order was never delivered and requested a refund"
- "Customer inquired about missing items from their completed order"
- "Customer requested cancellation of recurring subscription charges"
- "Customer reported delivery driver left groceries at wrong address"
- "Customer escalated complaint about damaged produce received in order"
"""

user_prompt_template = """Transcript:
{TRANSCRIPT}

Summarize the primary contact reason in ONE sentence (under 25 words):"""

In [6]:
# Check what columns you actually have
print("Column names in results_df:")
print(results_df.columns.tolist())
print("\nFirst row preview:")
print(results_df.head(1))

Column names in results_df:
['primary_contact_id', 'contact_channel', 'transcript_created_date_at_utc', 'is_retail_agent', 'subject', 'transcript']

First row preview:
   primary_contact_id contact_channel transcript_created_date_at_utc  \
0  500Uc00000Xsja9IAB           email            2025-05-01 19:21:25   

   is_retail_agent          subject  \
0                1  incomplete name   

                                          transcript  
0  Message 1 (2025-05-01 19:21:24.000 Z):\nThe br...  


In [8]:
def summarize_transcript(transcript, subject=None, contact_channel=None):
      """
      Summarize a single transcript into one sentence capturing the primary contact reason.
      Calls the AI Gateway via the `client` defined in Cell 5.
      """
      # Build the user prompt — append subject/channel context if available
      context = ""
      if contact_channel:
          context += f"Channel: {contact_channel}\n"
      if subject:
          context += f"Subject: {subject}\n"

      user_prompt = user_prompt_template.format(
          TRANSCRIPT=f"{context}{transcript}"
      )

      response = client.chat.completions.create(
          model="gpt-4o-2024-11-20",
          messages=[
              {"role": "system", "content": system_prompt},
              {"role": "user", "content": user_prompt},
          ],
          temperature=0.1,
          max_tokens=100,
      )

      return response.choices[0].message.content.strip()

In [9]:
# DIAGNOSTIC: Delete checkpoint and test ONE row manually
import os

# 1. Delete the bad checkpoint
checkpoint_file = 'cx_summary_checkpoint.csv'
if os.path.exists(checkpoint_file):
    os.remove(checkpoint_file)
    print(f"✅ Deleted bad checkpoint: {checkpoint_file}")

# 2. Test ONE row manually to see the real error
print("\n=== MANUAL TEST ON ONE ROW ===")
test_row = results_df.iloc[0]

print(f"Transcript length: {len(str(test_row['transcript']))}")
print(f"Subject: {test_row['subject']}")
print(f"Channel: {test_row['contact_channel']}")

try:
    # Test the API call
    result = summarize_transcript(
        transcript=test_row['transcript'],
        subject=test_row['subject'],
        contact_channel=test_row['contact_channel']
    )
    print(f"\n✅ SUCCESS! Result: {result}")
    
except Exception as e:
    print(f"\n❌ ERROR FOUND!")
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    import traceback
    print(f"\nFull traceback:")
    traceback.print_exc()


=== MANUAL TEST ON ONE ROW ===
Transcript length: 6766
Subject: incomplete name
Channel: email

✅ SUCCESS! Result: Retailer requested the addition of the brand name "Kalispell Kreamery" to the product information for a yogurt item.


In [10]:
# ============================================
# PROCESS ALL RECORDS WITH PARALLEL PROCESSING
# ============================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

def process_single_row(idx, row):
    """Process a single row with error handling"""
    try:
        # Handle potential missing data
        transcript = row.get('transcript')
        if not transcript or pd.isna(transcript):
            return idx, None, "No transcript available"
        
        subject = row.get('subject')
        channel = row.get('contact_channel')
        
        summary = summarize_transcript(transcript, subject, channel)
        return idx, summary, None
    except Exception as e:
        return idx, None, f"{type(e).__name__}: {str(e)}"


def summarize_parallel(df, max_workers=10, checkpoint_every=100, checkpoint_file='cx_summary_checkpoint.csv'):
    """
    Process transcripts in parallel with checkpointing
    
    Args:
        df: Input dataframe
        max_workers: Number of parallel threads (5-15 recommended)
        checkpoint_every: Save progress every N completions
        checkpoint_file: Where to save progress
    """
    
    # Check if checkpoint exists
    if os.path.exists(checkpoint_file):
        print(f"📁 Loading checkpoint from {checkpoint_file}")
        df = pd.read_csv(checkpoint_file)
        already_done = df['summary'].notna().sum()
        print(f"✅ Already completed: {already_done} rows")
    else:
        df = df.copy()
        df['summary'] = None
        df['error'] = None
    
    # Get rows that still need processing
    to_process = df[df['summary'].isna()]
    
    if len(to_process) == 0:
        print("🎉 All rows already processed!")
        return df
    
    print(f"🚀 Processing {len(to_process)} rows with {max_workers} parallel workers...")
    print(f"💾 Checkpointing every {checkpoint_every} completions to {checkpoint_file}")
    print(f"⏱️  Estimated time: {len(to_process) / (max_workers * 2) / 60:.1f} - {len(to_process) / (max_workers * 1) / 60:.1f} minutes\n")
    
    completed_count = 0
    start_time = time.time()
    
    # Submit all tasks to thread pool
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Create futures
        future_to_idx = {
            executor.submit(process_single_row, idx, row): idx 
            for idx, row in to_process.iterrows()
        }
        
        # Process as they complete
        for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="Summarizing"):
            idx, summary, error = future.result()
            
            if summary:
                df.at[idx, 'summary'] = summary
            if error:
                df.at[idx, 'error'] = error
                # Only print first few errors to avoid spam
                if completed_count < 5:
                    print(f"\n⚠️  Error on row {idx}: {error}")
            
            completed_count += 1
            
            # Checkpoint periodically
            if completed_count % checkpoint_every == 0:
                df.to_csv(checkpoint_file, index=False)
                elapsed = time.time() - start_time
                rate = completed_count / elapsed
                remaining = len(to_process) - completed_count
                eta_seconds = remaining / rate if rate > 0 else 0
                print(f"\n💾 Checkpoint saved. Progress: {completed_count}/{len(to_process)} ({completed_count/len(to_process)*100:.1f}%)")
                print(f"   Rate: {rate:.1f} rows/sec, ETA: {eta_seconds/60:.1f} min")
    
    # Final save
    df.to_csv(checkpoint_file, index=False)
    
    elapsed = time.time() - start_time
    success_count = df['summary'].notna().sum()
    error_count = df['error'].notna().sum()
    
    print(f"\n✅ COMPLETE! Processed {len(to_process)} rows in {elapsed/60:.1f} minutes")
    print(f"📊 Success: {success_count} ({success_count/len(df)*100:.1f}%)")
    print(f"⚠️  Errors: {error_count} ({error_count/len(df)*100:.1f}%)")
    print(f"⚡ Average rate: {len(to_process)/elapsed:.1f} rows/sec")
    
    return df


# ============================================
# RUN THE PROCESSING
# ============================================
print(f"📋 Total rows in results_df: {len(results_df):,}")
print(f"🎯 Starting parallel processing...\n")

# Process with 10 parallel workers (adjust if you hit rate limits)
results_with_summary = summarize_parallel(
    results_df, 
    max_workers=10,  # Reduce to 5 if you see rate limit errors
    checkpoint_every=100
)

print("\n" + "="*60)
print("SAMPLE RESULTS:")
print("="*60)
print(results_with_summary[['subject', 'summary', 'error']].head(20))


📋 Total rows in results_df: 36,598
🎯 Starting parallel processing...

🚀 Processing 36598 rows with 10 parallel workers...
💾 Checkpointing every 100 completions to cx_summary_checkpoint.csv
⏱️  Estimated time: 30.5 - 61.0 minutes



Summarizing:   0%|          | 25/36598 [00:00<05:03, 120.66it/s]


⚠️  Error on row 19: BadRequestError: Error code: 400 - {'statusCode': 400, 'errorType': 'Error', 'message': 'Error from OpenAI API: Bad Request: {\n  "error": {\n    "message": "This model\'s maximum context length is 128000 tokens. However, your messages resulted in 154725 tokens. Please reduce the length of the messages.",\n    "type": "invalid_request_error",\n    "param": "messages",\n    "code": "context_length_exceeded"\n  }\n}', 'error': {'message': "This model's maximum context length is 128000 tokens. However, your messages resulted in 154725 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}


Summarizing:   0%|          | 100/36598 [00:13<5:07:14,  1.98it/s]


💾 Checkpoint saved. Progress: 100/36598 (0.3%)
   Rate: 5.9 rows/sec, ETA: 102.6 min


...


💾 Checkpoint saved. Progress: 36400/36598 (99.5%)
   Rate: 13.5 rows/sec, ETA: 0.2 min


Summarizing: 100%|█████████▉| 36539/36598 [44:58<00:02, 20.99it/s]


💾 Checkpoint saved. Progress: 36500/36598 (99.7%)
   Rate: 13.5 rows/sec, ETA: 0.1 min


Summarizing: 100%|██████████| 36598/36598 [45:07<00:00, 13.52it/s]



✅ COMPLETE! Processed 36598 rows in 45.2 minutes
📊 Success: 36587 (100.0%)
⚠️  Errors: 11 (0.0%)
⚡ Average rate: 13.5 rows/sec

SAMPLE RESULTS:
                                              subject  \
0                                     incomplete name   
1   [EXTERNAL] Store Hours Change Tracker: Store#4...   
2   [EXTERNAL] Store Hours Change Tracker: Store#1...   
3   AP ROC Alert - Power Outage at Store 187 - Bay...   
4               URGENT: Store Closures & Deactivation   
5   AP ROC Alert - Power Outage at Store 046 - Bri...   
6                           Kendallville Kroger hours   
7                                         Store 22809   
8                                         add organic   
9   PSO-55242 Instacart Enterprise Technical Suppo...   
10                     Request for order cancellation   
11  [EXTERNAL] Store Hours Change Tracker: Store#7...   
12                          Store Name Update Request   
13        RV: Welcome to Instacart! - La Plaza Market   


In [11]:
# ============================================
# SAVE FINAL RESULTS
# ============================================

# Save to CSV for further analysis
output_file = 'cx_retailer_transcripts_summarized.csv'
results_with_summary.to_csv(output_file, index=False)
print(f"💾 Saved results to: {output_file}")


iq.upload(results_with_summary, "SANDBOX_DB_PII.SRIVIDYASEKAR.CX_RETAILER_CONTACTS_SUMMARY", if_exists="replace")

# Show statistics
print("\n" + "="*60)
print("FINAL STATISTICS:")
print("="*60)
print(f"Total records: {len(results_with_summary):,}")
print(f"Successfully summarized: {results_with_summary['summary'].notna().sum():,}")
print(f"Errors: {results_with_summary['error'].notna().sum():,}")
print(f"\nSuccess rate: {results_with_summary['summary'].notna().sum() / len(results_with_summary) * 100:.2f}%")

# Show error breakdown if any
if results_with_summary['error'].notna().sum() > 0:
    print("\n" + "="*60)
    print("ERROR BREAKDOWN:")
    print("="*60)
    print(results_with_summary['error'].value_counts().head(10))

# Preview successful summaries
print("\n" + "="*60)
print("SAMPLE SUCCESSFUL SUMMARIES:")
print("="*60)
successful = results_with_summary[results_with_summary['summary'].notna()].head(20)
for idx, row in successful.iterrows():
    print(f"\n{idx}. Channel: {row['contact_channel']}")
    if pd.notna(row['subject']):
        print(f"   Subject: {row['subject'][:80]}...")
    print(f"   Summary: {row['summary']}")


💾 Saved results to: cx_retailer_transcripts_summarized.csv

FINAL STATISTICS:
Total records: 36,598
Successfully summarized: 36,587
Errors: 11

Success rate: 99.97%

ERROR BREAKDOWN:
error
BadRequestError: Error code: 400 - {'statusCode': 400, 'errorType': 'Error', 'message': 'Error from OpenAI API: Bad Request: {\n  "error": {\n    "message": "This model\'s maximum context length is 128000 tokens. However, your messages resulted in 154725 tokens. Please reduce the length of the messages.",\n    "type": "invalid_request_error",\n    "param": "messages",\n    "code": "context_length_exceeded"\n  }\n}', 'error': {'message': "This model's maximum context length is 128000 tokens. However, your messages resulted in 154725 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}    1
BadRequestError: Error code: 400 - {'statusCode': 400, 'errorType': 'Error', 'message': 'Error from OpenAI API: Bad Request: {

# 🏗️ TAXONOMY BUILDING PIPELINE

This pipeline will:
1. **Step 1**: Load summarized data and explore patterns
2. **Step 2**: GPT-assisted L1 category discovery (3-6 top-level categories)
3. **Step 3**: Validate and refine L1 categories
4. **Step 4**: GPT-assisted L2 subcategory discovery for each L1
5. **Step 5**: Classify all 36k summaries into L1 categories
6. **Step 6**: Classify all summaries into L2 categories
7. **Step 7**: Review, validate, and export final taxonomy


# 📚 Quick Reference: How to Use This Pipeline

## Pipeline Overview
This notebook creates a hierarchical taxonomy (L1 → L2) for CX contact reasons using GPT-4 with **stratified sampling by channel** and a **holdout evaluation set** for unbiased validation.

## 🎯 Key Features
- **Stratified Sampling**: 70% phone/voice, 30% email in all sampling stages
- **Holdout Evaluation**: 500 clean samples never seen during discovery
- **Fast Validation**: Classify only 500 eval samples (~5-10 min) instead of all 36k
- **Checkpointing**: Auto-resume if interrupted

## Execution Order
Run cells in this order:

### Discovery Phase (Build Taxonomy)
1. **Cell 13**: Load summarized data → `df_valid` (36k summaries)
2. **Cell 14**: Discover L1 categories (700 phone + 300 email) → `l1_result`, `l1_used_indices`
3. **Cell 15**: Discover L2 subcategories (350 phone + 150 email per L1) → `l2_complete_taxonomy`, `l2_used_indices`

### Evaluation Phase (Test Taxonomy)
4. **Cell 16**: Import regex module (if needed)
5. **Cell 17**: Create holdout evaluation set (350 phone + 150 email, excluded from discovery) → `df_eval` (500 samples)
6. ⚠️ **BEFORE CLASSIFICATION**: Delete old checkpoints!
   ```python
   !rm -f classification_l1_checkpoint.csv classification_l2_checkpoint.csv
   !rm -f classification_l1_eval_checkpoint.csv classification_l2_eval_checkpoint.csv
   ```
7. **Cell 18/19**: Classify eval set into L1 (~2-5 min) → `df_with_l1`
8. **Cell 19/20**: Classify eval set into L2 (~2-5 min) → `df_with_l2`
9. **Cell 20/21**: Validate classification quality
10. **Cell 21/22**: Export final results

## 📊 Sample Sizes Summary

| Stage | Phone Samples | Email Samples | Total | Purpose |
|-------|--------------|---------------|-------|---------|
| **L1 Discovery** | 700 | 300 | 1,000 | Propose L1 categories |
| **L2 Discovery** | 350 per L1 | 150 per L1 | ~2,500 total | Propose L2 subcategories |
| **Holdout Eval** | 350 | 150 | 500 | Test taxonomy (zero overlap) |

## Output Files
- `taxonomy_l1_proposal.json` - Initial L1 category proposals
- `taxonomy_complete_proposal.json` - Complete L1+L2 taxonomy structure
- `classification_l1_eval_checkpoint.csv` - L1 classification results (eval set)
- `classification_l2_eval_checkpoint.csv` - L2 classification results (eval set)
- `cx_transcripts_with_taxonomy.csv` - Final classified data (if you run on full dataset)
- `cx_taxonomy_final.json` - Taxonomy definition for production use
- `cx_taxonomy_report.json` - Distribution statistics and summary

## ⚠️ Before Running Classification
**IMPORTANT**: Delete old checkpoint files to ensure fresh evaluation:
```python
!rm -f classification_*_checkpoint.csv classification_*_eval_checkpoint.csv
```
Or run the checkpoint cleanup cell before classification.

## Key Parameters to Adjust

### Sampling Strategy
- **L1 phone/email split**: Change `phone_samples=700, email_samples=300` in Cell 14
- **L2 phone/email split**: Change `phone_samples=350, email_samples=150` in Cell 15
- **Eval set size**: Adjust sampling in Cell 17 (currently 350 phone, 150 email)

### Taxonomy Size
- **L1 categories**: Change `num_l1_categories=6` in Cell 14
- **L2 subcategories**: Change `num_l2=5` in Cell 15

### Performance
- **Parallel workers**: Change `max_workers=10` if hitting rate limits (reduce to 5)
- **Checkpoint frequency**: Adjust `checkpoint_every=100` in classification calls

## Estimated Runtime

### Discovery Phase
- L1 Discovery: ~2 minutes (1,000 samples)
- L2 Discovery: ~5-10 minutes (~2,500 samples across all L1s)

### Evaluation Phase (500 samples)
- L1 Classification: ~2-5 minutes
- L2 Classification: ~2-5 minutes
- **Total eval time: ~10-20 minutes**

### Full Dataset Classification (if needed)
- L1 Classification: 30-60 minutes (36k records)
- L2 Classification: 30-60 minutes (36k records)

## 🔄 Iterating on Taxonomy

### Option 1: Refine and Re-Evaluate
1. Review validation output
2. Manually edit `l2_complete_taxonomy` JSON structure
3. Delete eval checkpoint files: `!rm classification_*_eval_checkpoint.csv`
4. Re-run classification cells on eval set

### Option 2: Rebuild from Scratch
1. Adjust parameters in Cells 14-15
2. Re-run discovery (Cells 14-15)
3. Create new eval set (Cell 17)
4. Delete all checkpoints
5. Re-run classification

## 🎯 Why Holdout Evaluation?
The 500-sample eval set:
- ✅ **Never seen during taxonomy discovery** (L1 or L2)
- ✅ **Unbiased validation** of taxonomy quality
- ✅ **Fast iteration** (~10 min vs 2 hours)
- ✅ **Representative** (70/30 phone/email split)
- ✅ **Clean for metrics** (precision, recall, confusion matrix)

## 📈 Scaling to Full Dataset
Once taxonomy is validated on eval set:
1. Modify classification cell to use `df_valid` instead of `df_eval`
2. Change checkpoint filenames (e.g., `classification_l1_full.csv`)
3. Increase `checkpoint_every` to 500 or 1000
4. Budget 1-2 hours for full classification


In [12]:
# ============================================
# STEP 1: LOAD SUMMARIZED DATA
# ============================================

# Load the summarized transcripts
summary_file = 'cx_retailer_transcripts_summarized.csv'
df_summaries = pd.read_csv(summary_file)

print(f"📊 Loaded {len(df_summaries):,} summarized transcripts")
print(f"✅ Valid summaries: {df_summaries['summary'].notna().sum():,}")
print(f"⚠️  Missing/Error summaries: {df_summaries['summary'].isna().sum():,}")

# Filter to only valid summaries for taxonomy building
df_valid = df_summaries[df_summaries['summary'].notna()].copy()
df_valid = df_valid[df_valid['summary'] != 'ERROR'].copy()

print(f"\n🎯 Working with {len(df_valid):,} valid summaries for taxonomy")

# Show sample
print("\n📋 Sample summaries:")
for i, row in df_valid.head(10).iterrows():
    print(f"{i+1}. {row['summary']}")


📊 Loaded 36,598 summarized transcripts
✅ Valid summaries: 36,587
⚠️  Missing/Error summaries: 11

🎯 Working with 36,587 valid summaries for taxonomy

📋 Sample summaries:
1. Retailer requested the addition of the brand name "Kalispell Kreamery" to the product information for a yogurt item.
2. Retailer reported early store closure due to severe weather, adjusting hours to close at 7 PM CST on 05/20/2025.
3. Customer reported a change in store hours for Store #171 due to limited staffing and a staff emergency.
4. Customer reported a power outage at Store 187 in Bay City, MI, impacting partial operations and requested further assistance.
5. Retailer requested deactivation of three store locations on specific dates due to upcoming closures and inquired about confirmation of the process.
6. Customer reported a power outage at Store 046 in Brighton, MI, and inquired about temporarily disabling deliveries during the outage.
7. Retailer reported incorrect pickup hours in the system and requeste

In [13]:
# ============================================
# STEP 2: GPT-ASSISTED L1 CATEGORY DISCOVERY
# ============================================

def discover_l1_categories(summaries, phone_samples=700, email_samples=300, num_l1_categories=4):
    """
    Use GPT to analyze summaries and propose L1 categories
    Uses stratified sampling by channel to ensure both phone and email are well-represented
    """
    # Stratified sampling by contact_channel
    # Get phone/voice contacts
    phone_data = summaries[summaries['contact_channel'].str.lower().isin(['phone', 'voice'])].copy()
    # Get email contacts
    email_data = summaries[summaries['contact_channel'].str.lower() == 'email'].copy()
    
    # Sample from each channel
    phone_sample_size = min(phone_samples, len(phone_data))
    email_sample_size = min(email_samples, len(email_data))
    
    phone_sample = phone_data.sample(n=phone_sample_size, random_state=42) if len(phone_data) > 0 else pd.DataFrame()
    email_sample = email_data.sample(n=email_sample_size, random_state=42) if len(email_data) > 0 else pd.DataFrame()
    
    # Combine samples
    combined_sample = pd.concat([phone_sample, email_sample], ignore_index=True)
    sample_summaries = combined_sample['summary'].tolist()
    used_indices = combined_sample.index.tolist()  # Track which rows were used
    
    print(f"🔍 Stratified sampling by channel:")
    print(f"   📞 Phone/Voice: {len(phone_sample)} samples (from {len(phone_data):,} total)")
    print(f"   📧 Email: {len(email_sample)} samples (from {len(email_data):,} total)")
    print(f"   📊 Total samples: {len(sample_summaries)}")
    
    # Format samples
    samples_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sample_summaries))
    
    # GPT prompt for L1 discovery
    response = client.chat.completions.create(
        model="gpt-4o-2024-11-20",
        temperature=0.3,
        messages=[
            {
                'role': 'system',
                'content': """You are an expert CX taxonomist building a contact reason classification system.
Your goal is to identify the top-level (L1) categories that represent the main reasons customers/retailers contact support.

Guidelines:
- Create 4-7 L1 categories that are mutually exclusive and collectively exhaustive
- Focus on the REASON for contact, not channel or outcome
- Use business-friendly names (e.g., "Account Management" not "Account Stuff")
- Each L1 should represent ~10-30% of contacts (avoid tiny or huge categories)
- Consider the voice of customer: what would they say they're contacting about?

Output format: JSON with L1 categories, descriptions, keywords, and estimated %"""
            },
            {
                'role': 'user',
                'content': f"""Analyze these {len(sample_summaries)} customer contact summaries and identify {num_l1_categories} top-level (L1) categories.

SUMMARIES:
{samples_text}

TASK:
1. Identify {num_l1_categories} L1 categories
2. For each L1, provide:
   - Clear name
   - Business description (1 sentence)
   - Top 20 keywords/phrases
   - Estimated % of total contacts
   - 5 example IDs from the list above

Output JSON in this exact schema:
{{
  "L1_categories": [
    {{
      "id": "L1_01",
      "name": "Category Name",
      "description": "What this category covers",
      "keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5", "keyword6", "keyword7", "keyword8", "keyword9", "keyword10", "keyword11", "keyword12", "keyword13", "keyword14", "keyword15", "keyword16", "keyword17", "keyword18", "keyword19", "keyword20"],
      "estimated_pct": 25,
      "example_ids": [1, 15, 203, 456, 789]
    }}
  ],
  "analysis_notes": "Any observations about the data"
}}"""
            }
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return result, sample_summaries, used_indices


# Run L1 discovery
print("🚀 Starting L1 category discovery...")
l1_result, sample_summaries, l1_used_indices = discover_l1_categories(df_valid, phone_samples=700, email_samples=300, num_l1_categories=6)

print(f"\n📝 Tracking: {len(l1_used_indices)} indices used in L1 discovery")

# Display results
print("\n" + "="*80)
print("📊 PROPOSED L1 CATEGORIES")
print("="*80)

for cat in l1_result['L1_categories']:
    print(f"\n{cat['id']}: {cat['name']} (~{cat['estimated_pct']}%)")
    print(f"   Description: {cat['description']}")
    print(f"   Keywords: {', '.join(cat['keywords'][:5])}...")
    print(f"   Examples:")
    for ex_id in cat['example_ids'][:3]:
        if ex_id <= len(sample_summaries):
            print(f"      - {sample_summaries[ex_id-1]}")

if 'analysis_notes' in l1_result:
    print(f"\n📝 Analysis Notes: {l1_result['analysis_notes']}")

# Save L1 proposal
with open('taxonomy_l1_proposal.json', 'w') as f:
    json.dump(l1_result, f, indent=2)
print("\n💾 Saved L1 proposal to: taxonomy_l1_proposal.json")


🚀 Starting L1 category discovery...
🔍 Stratified sampling by channel:
   📞 Phone/Voice: 700 samples (from 32,625 total)
   📧 Email: 300 samples (from 3,962 total)
   📊 Total samples: 1000

📝 Tracking: 1000 indices used in L1 discovery

📊 PROPOSED L1 CATEGORIES

L1_01: Order Status and Delivery Issues (~30%)
   Description: Covers inquiries related to order status, delivery delays, missing or incorrect deliveries, and related updates.
   Keywords: order status, delivery delay, missing order, incorrect delivery, delivery status...
   Examples:
      - Retailer inquired about the status of a delayed order on behalf of a customer.
      - Retailer inquired about the delivery status of a customer's order.
      - Retailer reported a customer's groceries were delivered to the wrong address and requested assistance in resolving the delivery issue.

L1_02: Order Modifications and Cancellations (~25%)
   Description: Covers requests to modify or cancel orders due to customer changes, unavailabl

In [14]:
# ============================================
# STEP 3: GPT-ASSISTED L2 SUBCATEGORY DISCOVERY
# ============================================

def discover_l2_categories(l1_category, summaries, phone_samples=350, email_samples=150, num_l2=5):
    """
    For a given L1 category, use GPT to propose L2 subcategories
    Uses stratified sampling by channel (70% phone, 30% email by default)
    With improved fallback: blends keyword matches with random samples when insufficient matches
    """
    # Filter summaries that likely belong to this L1 based on keywords
    keywords = l1_category['keywords']
    keyword_pattern = '|'.join([re.escape(k.lower()) for k in keywords[:10]])
    
    # Find summaries matching this L1's keywords
    matching = summaries[summaries['summary'].str.lower().str.contains(keyword_pattern, na=False, regex=True)]
    
    print(f"🔍 Discovering L2 subcategories for: {l1_category['name']}")
    print(f"   🔎 Keyword matches found: {len(matching)}")
    
    # Split matches by channel
    phone_matches = matching[matching['contact_channel'].str.lower().isin(['phone', 'voice'])]
    email_matches = matching[matching['contact_channel'].str.lower() == 'email']
    
    print(f"      Phone matches: {len(phone_matches)}, Email matches: {len(email_matches)}")
    
    # Check if we have enough matches per channel
    if len(phone_matches) < phone_samples or len(email_matches) < email_samples:
        print(f"   ⚠️  Insufficient keyword matches")
        print(f"   💡 Blending keyword matches with random samples...")
        
        # Use all keyword matches
        phone_sample = phone_matches.copy()
        email_sample = email_matches.copy()
        
        # Fill remaining with random samples (excluding keyword matches)
        if len(phone_sample) < phone_samples:
            phone_pool = summaries[
                (summaries['contact_channel'].str.lower().isin(['phone', 'voice'])) &
                (~summaries.index.isin(phone_matches.index))
            ]
            additional_needed = phone_samples - len(phone_sample)
            if len(phone_pool) > 0:
                additional_phone = phone_pool.sample(
                    n=min(additional_needed, len(phone_pool)),
                    random_state=42
                )
                phone_sample = pd.concat([phone_sample, additional_phone])
        
        if len(email_sample) < email_samples:
            email_pool = summaries[
                (summaries['contact_channel'].str.lower() == 'email') &
                (~summaries.index.isin(email_matches.index))
            ]
            additional_needed = email_samples - len(email_sample)
            if len(email_pool) > 0:
                additional_email = email_pool.sample(
                    n=min(additional_needed, len(email_pool)),
                    random_state=42
                )
                email_sample = pd.concat([email_sample, additional_email])
        
        # Track how many from keywords vs random
        phone_from_keywords = len([i for i in phone_sample.index if i in phone_matches.index])
        email_from_keywords = len([i for i in email_sample.index if i in email_matches.index])
        
        print(f"   📞 Phone: {len(phone_sample)} total ({phone_from_keywords} keywords + {len(phone_sample)-phone_from_keywords} random)")
        print(f"   📧 Email: {len(email_sample)} total ({email_from_keywords} keywords + {len(email_sample)-email_from_keywords} random)")
    else:
        # Enough matches, sample from keyword matches only
        phone_sample = phone_matches.sample(n=min(phone_samples, len(phone_matches)), random_state=42)
        email_sample = email_matches.sample(n=min(email_samples, len(email_matches)), random_state=42)
        
        print(f"   ✅ Sufficient keyword matches")
        print(f"   📞 Phone: {len(phone_sample)} samples (all from keywords)")
        print(f"   📧 Email: {len(email_sample)} samples (all from keywords)")
    
    # Combine samples
    combined_sample = pd.concat([phone_sample, email_sample])
    sample_summaries = combined_sample['summary'].tolist()
    used_indices = combined_sample.index.tolist()
    
    sample_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sample_summaries))
    
    print(f"   📊 Total samples for L2 discovery: {len(sample_summaries)}")
    
    # Rest of the function stays the same (GPT call)
    response = client.chat.completions.create(
        model="gpt-4o-2024-11-20",
        temperature=0.3,
        messages=[
            {
                'role': 'system',
                'content': """You are an expert CX taxonomist creating detailed subcategories (L2) within a top-level category (L1).

Guidelines:
- Create 3-7 L2 subcategories that are mutually exclusive within this L1
- L2 categories should be specific and actionable
- Each L2 should represent a distinct contact reason or issue type
- Use clear, business-friendly names
- Focus on what would help teams route and prioritize contacts"""
            },
            {
                'role': 'user',
                'content': f"""You are creating L2 subcategories for the L1 category: "{l1_category['name']}"

L1 Description: {l1_category['description']}

Here are {len(sample_summaries)} contact summaries that belong to this L1 category:

{sample_text}

TASK:
Create {num_l2} L2 subcategories that divide "{l1_category['name']}" into meaningful, specific groups.

Output JSON in this schema:
{{
  "L2_categories": [
    {{
      "id": "L2_01",
      "name": "Subcategory Name",
      "description": "What this subcategory covers",
      "keywords": ["keyword1", "keyword2", ...],
      "example_ids": [1, 5, 12]
    }}
  ]
}}"""
            }
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return result, matching['summary'].tolist(), used_indices


# Discover L2 for each L1
print("🚀 Starting L2 subcategory discovery for each L1...")
print("="*80)

l2_complete_taxonomy = {
    'L1_categories': []
}

l2_used_indices = []  # Track all indices used across all L2 discoveries

for l1_cat in l1_result['L1_categories']:
    l2_result, l2_samples, l2_indices = discover_l2_categories(l1_cat, df_valid, phone_samples=350, email_samples=150, num_l2=8)
    l2_used_indices.extend(l2_indices)  # Accumulate indices
    
    # Add L2 categories to the L1 category
    l1_with_l2 = l1_cat.copy()
    l1_with_l2['L2_categories'] = l2_result['L2_categories']
    
    # Update L2 IDs to include L1 prefix
    for i, l2_cat in enumerate(l1_with_l2['L2_categories']):
        l2_cat['id'] = f"{l1_cat['id']}_L2_{i+1:02d}"
    
    l2_complete_taxonomy['L1_categories'].append(l1_with_l2)
    
    # Display
    print(f"\n✅ {l1_cat['name']}:")
    for l2_cat in l1_with_l2['L2_categories']:
        print(f"   └─ {l2_cat['id']}: {l2_cat['name']}")
        print(f"      {l2_cat['description']}")
        print(f"      Keywords: {', '.join(l2_cat['keywords'][:3])}...")

# Save complete taxonomy
with open('taxonomy_complete_proposal.json', 'w') as f:
    json.dump(l2_complete_taxonomy, f, indent=2)

print("\n" + "="*80)
print("💾 Saved complete L1+L2 taxonomy to: taxonomy_complete_proposal.json")
print("="*80)

# Track all indices used in discovery
all_discovery_indices = set(l1_used_indices + l2_used_indices)
print(f"\n📝 Discovery dataset tracking:")
print(f"   L1 discovery: {len(l1_used_indices)} unique indices")
print(f"   L2 discovery: {len(set(l2_used_indices))} unique indices (across all L1s)")
print(f"   Combined unique: {len(all_discovery_indices)} indices used in discovery")
print(f"   Available for holdout: {len(df_valid) - len(all_discovery_indices):,} indices")


🚀 Starting L2 subcategory discovery for each L1...
🔍 Discovering L2 subcategories for: Order Status and Delivery Issues
   🔎 Keyword matches found: 32853
      Phone matches: 31043, Email matches: 1810
   ✅ Sufficient keyword matches
   📞 Phone: 350 samples (all from keywords)
   📧 Email: 150 samples (all from keywords)
   📊 Total samples for L2 discovery: 500

✅ Order Status and Delivery Issues:
   └─ L1_01_L2_01: Order Status Inquiries
      Covers requests for updates on the status of an order, including whether it has been assigned, picked, or delivered.
      Keywords: order status, order update, delivery time...
   └─ L1_01_L2_02: Delivery Delays
      Covers issues related to delayed deliveries, including inquiries about reasons for delays and requests for updated delivery times.
      Keywords: delayed delivery, delivery delay, late order...
   └─ L1_01_L2_03: Missing or Incorrect Deliveries
      Covers reports of orders not being delivered, delivered to the wrong address, or 

In [15]:
# ============================================
# CREATE HOLDOUT EVALUATION SET
# ============================================

print("🎯 Creating holdout evaluation set...")
print("="*80)

# Get all indices NOT used in L1 or L2 discovery
holdout_pool = df_valid[~df_valid.index.isin(all_discovery_indices)].copy()

print(f"📊 Holdout pool size: {len(holdout_pool):,} (excluded {len(all_discovery_indices)} discovery samples)")

# Stratified sampling from holdout pool: 350 phone, 150 email
phone_holdout = holdout_pool[holdout_pool['contact_channel'].str.lower().isin(['phone', 'voice'])].copy()
email_holdout = holdout_pool[holdout_pool['contact_channel'].str.lower() == 'email'].copy()

# Sample from each channel
phone_eval_size = min(700, len(phone_holdout))
email_eval_size = min(300, len(email_holdout))

phone_eval = phone_holdout.sample(n=phone_eval_size, random_state=99) if len(phone_holdout) > 0 else pd.DataFrame()
email_eval = email_holdout.sample(n=email_eval_size, random_state=99) if len(email_holdout) > 0 else pd.DataFrame()

# Combine into evaluation set
df_eval = pd.concat([phone_eval, email_eval], ignore_index=False).copy()

print(f"\n✅ Holdout evaluation set created:")
print(f"   📞 Phone/Voice: {len(phone_eval)} samples")
print(f"   📧 Email: {len(email_eval)} samples")
print(f"   📊 Total evaluation set: {len(df_eval)} samples")
print(f"\n🔒 Verification:")
print(f"   Overlap with L1 discovery: {len(set(df_eval.index) & set(l1_used_indices))}")
print(f"   Overlap with L2 discovery: {len(set(df_eval.index) & set(l2_used_indices))}")
print(f"   ✓ All overlaps should be 0!")

# Show sample
print(f"\n📋 Sample from evaluation set:")
for idx, row in df_eval.head(5).iterrows():
    print(f"   {idx}. [{row['contact_channel']}] {row['summary'][:80]}...")


🎯 Creating holdout evaluation set...
📊 Holdout pool size: 30,817 (excluded 5771 discovery samples)

✅ Holdout evaluation set created:
   📞 Phone/Voice: 700 samples
   📧 Email: 300 samples
   📊 Total evaluation set: 1000 samples

🔒 Verification:
   Overlap with L1 discovery: 0
   Overlap with L2 discovery: 0
   ✓ All overlaps should be 0!

📋 Sample from evaluation set:
   36435. [phone] Retailer reported customer complaint about a rude delivery driver and requested ...
   12892. [phone] Retailer inquired about the status of a customer's order on their behalf....
   23211. [phone] Retailer inquired about the delivery status and estimated time of arrival for a ...
   10206. [phone] Retailer inquired about the reason a shopper was banned from their store and req...
   31066. [phone] Retailer requested cancellation of a customer's order on their behalf....


In [16]:
# ============================================
# CLEAN UP OLD CHECKPOINTS (Run once)
# ============================================

import os

# Delete old checkpoint files to ensure fresh eval classification
checkpoint_files = [
    'classification_l1_checkpoint.csv',
    'classification_l2_checkpoint.csv',
    'classification_l1_eval_checkpoint.csv',
    'classification_l2_eval_checkpoint.csv'
]

for file in checkpoint_files:
    if os.path.exists(file):
        os.remove(file)
        print(f"🗑️  Deleted: {file}")
    else:
        print(f"✓ Not found: {file}")

print("\n✅ Checkpoint cleanup complete!")

✓ Not found: classification_l1_checkpoint.csv
✓ Not found: classification_l2_checkpoint.csv
✓ Not found: classification_l1_eval_checkpoint.csv
✓ Not found: classification_l2_eval_checkpoint.csv

✅ Checkpoint cleanup complete!


In [17]:
# ============================================
# STEP 4: CLASSIFY Holdout evaluation SUMMARIES INTO L1 CATEGORIES
# ============================================

def classify_to_l1(summary, taxonomy):
    """
    Classify a single summary into an L1 category using GPT
    """
    # Build category list for prompt
    category_list = []
    for l1 in taxonomy['L1_categories']:
        category_list.append(f"{l1['id']}: {l1['name']} - {l1['description']}")
    
    categories_text = "\n".join(category_list)
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-11-20",
        temperature=0.1,  # Low temperature for consistent classification
        max_tokens=50,
        messages=[
            {
                'role': 'system',
                'content': """You are a contact reason classifier. Given a summary, pick the most appropriate L1 category.
                
Rules:
- Output ONLY the category ID (e.g., "L1_01")
- Be consistent and precise
- If genuinely uncertain, pick the closest match"""
            },
            {
                'role': 'user',
                'content': f"""Categories:
{categories_text}

Summary: {summary}

Which L1 category best fits this summary? Output only the ID (e.g., "L1_01"):"""
            }
        ]
    )
    
    return response.choices[0].message.content.strip()


def classify_l1_parallel(df, taxonomy, max_workers=10, checkpoint_file='classification_l1_checkpoint.csv'):
    """
    Classify all summaries into L1 categories in parallel
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm import tqdm
    import time
    
    # Check if checkpoint exists
    if os.path.exists(checkpoint_file):
        print(f"📁 Loading checkpoint from {checkpoint_file}")
        df = pd.read_csv(checkpoint_file)
        already_done = df['L1_category'].notna().sum()
        print(f"✅ Already classified: {already_done} rows")
    else:
        df = df.copy()
        df['L1_category'] = None
        df['L1_confidence'] = None
    
    # Get rows that still need classification
    to_process = df[df['L1_category'].isna()]
    
    if len(to_process) == 0:
        print("🎉 All rows already classified!")
        return df
    
    print(f"🚀 Classifying {len(to_process):,} summaries into L1 categories...")
    print(f"⏱️  Estimated time: {len(to_process) / (max_workers * 2) / 60:.1f} - {len(to_process) / max_workers / 60:.1f} minutes\n")
    
    def process_row(idx, row):
        try:
            l1_cat = classify_to_l1(row['summary'], taxonomy)
            return idx, l1_cat, None
        except Exception as e:
            return idx, None, str(e)
    
    completed_count = 0
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(process_row, idx, row): idx 
            for idx, row in to_process.iterrows()
        }
        
        for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="L1 Classification"):
            idx, l1_cat, error = future.result()
            
            if l1_cat:
                df.at[idx, 'L1_category'] = l1_cat
            if error and completed_count < 5:
                print(f"\n⚠️  Error on row {idx}: {error}")
            
            completed_count += 1
            
            # Checkpoint every 500 rows
            if completed_count % 500 == 0:
                df.to_csv(checkpoint_file, index=False)
                elapsed = time.time() - start_time
                rate = completed_count / elapsed
                remaining = len(to_process) - completed_count
                eta_seconds = remaining / rate if rate > 0 else 0
                print(f"\n💾 Checkpoint: {completed_count}/{len(to_process)} ({completed_count/len(to_process)*100:.1f}%), ETA: {eta_seconds/60:.1f}min")
    
    # Final save
    df.to_csv(checkpoint_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"\n✅ L1 Classification complete! ({elapsed/60:.1f} minutes)")
    
    return df


# Run L1 classification ON EVALUATION SET ONLY
print("🎯 Starting L1 classification for EVALUATION SET ONLY...")
print(f"   Classifying {len(df_eval)} holdout samples (not used in discovery)\n")
df_with_l1 = classify_l1_parallel(df_eval, l2_complete_taxonomy, max_workers=10)

# Show distribution
print("\n" + "="*80)
print("📊 L1 CATEGORY DISTRIBUTION")
print("="*80)
l1_dist = df_with_l1['L1_category'].value_counts()
for cat_id, count in l1_dist.items():
    # Find category name
    cat_name = next((c['name'] for c in l2_complete_taxonomy['L1_categories'] if c['id'] == cat_id), cat_id)
    print(f"{cat_id}: {cat_name:30s} - {count:6,} ({count/len(df_with_l1)*100:5.1f}%)")


🎯 Starting L1 classification for EVALUATION SET ONLY...
   Classifying 1000 holdout samples (not used in discovery)

🚀 Classifying 1,000 summaries into L1 categories...
⏱️  Estimated time: 0.8 - 1.7 minutes



L1 Classification:  50%|█████     | 505/1000 [00:22<00:22, 22.43it/s]


💾 Checkpoint: 500/1000 (50.0%), ETA: 0.4min


L1 Classification: 100%|██████████| 1000/1000 [00:46<00:00, 21.48it/s]


💾 Checkpoint: 1000/1000 (100.0%), ETA: 0.0min

✅ L1 Classification complete! (0.8 minutes)

📊 L1 CATEGORY DISTRIBUTION
L1_06: Technical and System Issues    -    312 ( 31.2%)
L1_01: Order Status and Delivery Issues -    253 ( 25.3%)
L1_02: Order Modifications and Cancellations -    200 ( 20.0%)
L1_03: Refunds and Compensation       -     86 (  8.6%)
L1_05: Shopper and Driver Issues      -     79 (  7.9%)
L1_04: Payment and Transaction Issues -     70 (  7.0%)


In [18]:
# ============================================
# STEP 5: CLASSIFY ALL SUMMARIES INTO L2 CATEGORIES
# ============================================

def classify_to_l2(summary, l1_category, taxonomy):
    """
    Classify a summary into an L2 category given its L1
    """
    # Find the L1 category details
    l1_cat = next((c for c in taxonomy['L1_categories'] if c['id'] == l1_category), None)
    
    if not l1_cat or 'L2_categories' not in l1_cat:
        return None
    
    # Build L2 category list for this L1
    l2_list = []
    for l2 in l1_cat['L2_categories']:
        l2_list.append(f"{l2['id']}: {l2['name']} - {l2['description']}")
    
    l2_text = "\n".join(l2_list)
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-11-20",
        temperature=0.1,
        max_tokens=50,
        messages=[
            {
                'role': 'system',
                'content': """You are a contact reason classifier. Given a summary and its L1 category, pick the most appropriate L2 subcategory.

Rules:
- Output ONLY the L2 category ID (e.g., "L1_01_L2_03")
- Be consistent and precise"""
            },
            {
                'role': 'user',
                'content': f"""L1 Category: {l1_cat['name']}

L2 Subcategories:
{l2_text}

Summary: {summary}

Which L2 subcategory best fits? Output only the ID:"""
            }
        ]
    )
    
    return response.choices[0].message.content.strip()


def classify_l2_parallel(df, taxonomy, max_workers=10, checkpoint_file='classification_l2_checkpoint.csv'):
    """
    Classify all summaries into L2 categories based on their L1
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm import tqdm
    import time
    
    # Check if checkpoint exists
    if os.path.exists(checkpoint_file):
        print(f"📁 Loading checkpoint from {checkpoint_file}")
        df = pd.read_csv(checkpoint_file)
        already_done = df['L2_category'].notna().sum()
        print(f"✅ Already classified: {already_done} rows")
    else:
        df = df.copy()
        df['L2_category'] = None
    
    # Get rows that still need L2 classification (must have L1 first)
    to_process = df[(df['L1_category'].notna()) & (df['L2_category'].isna())]
    
    if len(to_process) == 0:
        print("🎉 All rows already classified!")
        return df
    
    print(f"🚀 Classifying {len(to_process):,} summaries into L2 subcategories...")
    print(f"⏱️  Estimated time: {len(to_process) / (max_workers * 2) / 60:.1f} - {len(to_process) / max_workers / 60:.1f} minutes\n")
    
    def process_row(idx, row):
        try:
            l2_cat = classify_to_l2(row['summary'], row['L1_category'], taxonomy)
            return idx, l2_cat, None
        except Exception as e:
            return idx, None, str(e)
    
    completed_count = 0
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(process_row, idx, row): idx 
            for idx, row in to_process.iterrows()
        }
        
        for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="L2 Classification"):
            idx, l2_cat, error = future.result()
            
            if l2_cat:
                df.at[idx, 'L2_category'] = l2_cat
            if error and completed_count < 5:
                print(f"\n⚠️  Error on row {idx}: {error}")
            
            completed_count += 1
            
            # Checkpoint every 500 rows
            if completed_count % 500 == 0:
                df.to_csv(checkpoint_file, index=False)
                elapsed = time.time() - start_time
                rate = completed_count / elapsed
                remaining = len(to_process) - completed_count
                eta_seconds = remaining / rate if rate > 0 else 0
                print(f"\n💾 Checkpoint: {completed_count}/{len(to_process)} ({completed_count/len(to_process)*100:.1f}%), ETA: {eta_seconds/60:.1f}min")
    
    # Final save
    df.to_csv(checkpoint_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"\n✅ L2 Classification complete! ({elapsed/60:.1f} minutes)")
    
    return df


# Run L2 classification
print("🎯 Starting L2 classification for all summaries...")
df_with_l2 = classify_l2_parallel(df_with_l1, l2_complete_taxonomy, max_workers=10)

# Show L2 distribution within each L1
print("\n" + "="*80)
print("📊 L2 CATEGORY DISTRIBUTION (within each L1)")
print("="*80)

for l1_cat in l2_complete_taxonomy['L1_categories']:
    l1_data = df_with_l2[df_with_l2['L1_category'] == l1_cat['id']]
    
    if len(l1_data) == 0:
        continue
    
    print(f"\n{l1_cat['id']}: {l1_cat['name']} (Total: {len(l1_data):,})")
    l2_dist = l1_data['L2_category'].value_counts()
    
    for l2_id, count in l2_dist.items():
        # Find L2 name
        l2_name = next((l2['name'] for l2 in l1_cat.get('L2_categories', []) if l2['id'] == l2_id), l2_id)
        print(f"   └─ {l2_id}: {l2_name:35s} - {count:5,} ({count/len(l1_data)*100:5.1f}%)")


🎯 Starting L2 classification for all summaries...
🚀 Classifying 1,000 summaries into L2 subcategories...
⏱️  Estimated time: 0.8 - 1.7 minutes



L2 Classification:  50%|█████     | 504/1000 [00:23<00:23, 21.50it/s]


💾 Checkpoint: 500/1000 (50.0%), ETA: 0.4min


L2 Classification: 100%|██████████| 1000/1000 [00:48<00:00, 20.73it/s]


💾 Checkpoint: 1000/1000 (100.0%), ETA: 0.0min

✅ L2 Classification complete! (0.8 minutes)

📊 L2 CATEGORY DISTRIBUTION (within each L1)

L1_01: Order Status and Delivery Issues (Total: 253)
   └─ L1_01_L2_01: Order Status Inquiries              -    97 ( 38.3%)
   └─ L1_01_L2_03: Missing or Incorrect Deliveries     -    94 ( 37.2%)
   └─ L1_01_L2_02: Delivery Delays                     -    60 ( 23.7%)
   └─ L1_01_L2_08: Order Modifications                 -     1 (  0.4%)
   └─ L1_01_L2_06: Technical or App Issues             -     1 (  0.4%)

L1_02: Order Modifications and Cancellations (Total: 200)
   └─ L1_02_L2_04: Customer-Initiated Cancellations    -    86 ( 43.0%)
   └─ L1_02_L2_02: Unavailable Items Cancellations     -    47 ( 23.5%)
   └─ L1_02_L2_08: Order Status and Verification       -    26 ( 13.0%)
   └─ L1_02_L2_05: System or Technical Issues          -    22 ( 11.0%)
   └─ L1_02_L2_07: Shopper or Driver Issues            -     8 (  4.0%)
   └─ L1_02_L2_06: Payment and

In [19]:
# ============================================
# STEP 6: VALIDATION & QUALITY REVIEW
# ============================================

print("🔍 TAXONOMY QUALITY REVIEW")
print("="*80)

# 1. Check classification completeness
total_records = len(df_with_l2)
l1_classified = df_with_l2['L1_category'].notna().sum()
l2_classified = df_with_l2['L2_category'].notna().sum()

print(f"\n📊 Classification Completeness:")
print(f"   Total records: {total_records:,}")
print(f"   L1 classified: {l1_classified:,} ({l1_classified/total_records*100:.1f}%)")
print(f"   L2 classified: {l2_classified:,} ({l2_classified/total_records*100:.1f}%)")

# 2. Check for category balance
print(f"\n⚖️  Category Balance Check:")
l1_dist = df_with_l2['L1_category'].value_counts()
for cat_id, count in l1_dist.items():
    pct = count / total_records * 100
    cat_name = next((c['name'] for c in l2_complete_taxonomy['L1_categories'] if c['id'] == cat_id), cat_id)
    status = "✅" if 5 <= pct <= 40 else "⚠️ "
    print(f"   {status} {cat_name:30s}: {pct:5.1f}%")

# 3. Sample validation - show random examples from each L1
print(f"\n📋 Sample Classifications (5 random per L1):")
for l1_cat in l2_complete_taxonomy['L1_categories']:
    l1_data = df_with_l2[df_with_l2['L1_category'] == l1_cat['id']]
    
    if len(l1_data) == 0:
        continue
    
    print(f"\n{l1_cat['name']}:")
    samples = l1_data.sample(min(5, len(l1_data)))
    
    for idx, row in samples.iterrows():
        l2_name = "N/A"
        if pd.notna(row.get('L2_category')):
            l2 = next((l2 for l2 in l1_cat.get('L2_categories', []) if l2['id'] == row['L2_category']), None)
            if l2:
                l2_name = l2['name']
        
        print(f"   └─ L2: {l2_name}")
        print(f"      Summary: {row['summary'][:100]}...")

# 4. Add category names to dataframe for export
def add_category_names(df, taxonomy):
    """Add human-readable category names"""
    df = df.copy()
    
    # Add L1 names
    l1_map = {c['id']: c['name'] for c in taxonomy['L1_categories']}
    df['L1_category_name'] = df['L1_category'].map(l1_map)
    
    # Add L2 names
    l2_map = {}
    for l1_cat in taxonomy['L1_categories']:
        for l2_cat in l1_cat.get('L2_categories', []):
            l2_map[l2_cat['id']] = l2_cat['name']
    df['L2_category_name'] = df['L2_category'].map(l2_map)
    
    return df

df_final = add_category_names(df_with_l2, l2_complete_taxonomy)

print("\n✅ Validation complete! Ready for export.")


🔍 TAXONOMY QUALITY REVIEW

📊 Classification Completeness:
   Total records: 1,000
   L1 classified: 1,000 (100.0%)
   L2 classified: 1,000 (100.0%)

⚖️  Category Balance Check:
   ✅ Technical and System Issues   :  31.2%
   ✅ Order Status and Delivery Issues:  25.3%
   ✅ Order Modifications and Cancellations:  20.0%
   ✅ Refunds and Compensation      :   8.6%
   ✅ Shopper and Driver Issues     :   7.9%
   ✅ Payment and Transaction Issues:   7.0%

📋 Sample Classifications (5 random per L1):

Order Status and Delivery Issues:
   └─ L2: Order Status Inquiries
      Summary: Retailer reported customer wanted to know the status of their order, as the shopper had not left for...
   └─ L2: Order Modifications
      Summary: Retailer requested assistance in rescheduling a customer's delivery time to an earlier slot due to a...
   └─ L2: Delivery Delays
      Summary: Retailer inquired about the delayed order status on behalf of a customer and requested the reason fo...
   └─ L2: Missing or Inc

In [20]:
# ============================================
# STEP 7: EXPORT FINAL TAXONOMY & CLASSIFIED DATA
# ============================================

print("💾 EXPORTING FINAL RESULTS")
print("="*80)

# 1. Export classified data with all fields
output_file = 'cx_transcripts_with_taxonomy.csv'
df_final.to_csv(output_file, index=False)
print(f"\n✅ Exported classified data to: {output_file}")
print(f"   Columns: {', '.join(df_final.columns)}")

# 2. Export taxonomy definition
taxonomy_file = 'cx_taxonomy_final.json'
with open(taxonomy_file, 'w') as f:
    json.dump(l2_complete_taxonomy, f, indent=2)
print(f"\n✅ Exported taxonomy definition to: {taxonomy_file}")

# 3. Create summary report
summary_report = {
    'metadata': {
        'total_records': len(df_final),
        'date_created': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'channels': df_final['contact_channel'].value_counts().to_dict()
    },
    'L1_distribution': {},
    'L2_distribution': {}
}

# L1 distribution
for l1_cat in l2_complete_taxonomy['L1_categories']:
    l1_data = df_final[df_final['L1_category'] == l1_cat['id']]
    count = len(l1_data)
    
    summary_report['L1_distribution'][l1_cat['id']] = {
        'name': l1_cat['name'],
        'description': l1_cat['description'],
        'count': count,
        'percentage': round(count / len(df_final) * 100, 2)
    }
    
    # L2 distribution within this L1
    l2_dist = {}
    for l2_cat in l1_cat.get('L2_categories', []):
        l2_count = len(df_final[df_final['L2_category'] == l2_cat['id']])
        if l2_count > 0:
            l2_dist[l2_cat['id']] = {
                'name': l2_cat['name'],
                'count': l2_count,
                'percentage_within_l1': round(l2_count / max(count, 1) * 100, 2)
            }
    
    summary_report['L2_distribution'][l1_cat['id']] = l2_dist

report_file = 'cx_taxonomy_report.json'
with open(report_file, 'w') as f:
    json.dump(summary_report, f, indent=2)
print(f"\n✅ Exported summary report to: {report_file}")

# 4. Create human-readable summary
print("\n" + "="*80)
print("📊 FINAL TAXONOMY SUMMARY")
print("="*80)
print(f"\nTotal Contacts Classified: {len(df_final):,}")
print(f"Classification Rate: {df_final['L2_category'].notna().sum() / len(df_final) * 100:.1f}%")

print(f"\n🏷️  L1 Categories ({len(l2_complete_taxonomy['L1_categories'])}):")
for l1_id, l1_data in summary_report['L1_distribution'].items():
    print(f"\n   {l1_id}: {l1_data['name']} ({l1_data['percentage']:.1f}%)")
    print(f"   {l1_data['description']}")
    
    # Show L2s
    if l1_id in summary_report['L2_distribution']:
        l2_cats = summary_report['L2_distribution'][l1_id]
        print(f"   L2 Subcategories ({len(l2_cats)}):")
        for l2_id, l2_data in l2_cats.items():
            print(f"      └─ {l2_data['name']}: {l2_data['percentage_within_l1']:.1f}% of {l1_data['name']}")

print("\n" + "="*80)
print("✨ TAXONOMY BUILD COMPLETE!")
print("="*80)
print("\nFiles created:")
print(f"1. {output_file}")
print(f"2. {taxonomy_file}")
print(f"3. {report_file}")
print("\nNext steps:")
print("- Review sample classifications for quality")
print("- Refine categories if needed and re-run classification")
print("- Use taxonomy for reporting, routing, and analysis")


💾 EXPORTING FINAL RESULTS

✅ Exported classified data to: cx_transcripts_with_taxonomy.csv
   Columns: primary_contact_id, contact_channel, transcript_created_date_at_utc, is_retail_agent, subject, transcript, summary, error, L1_category, L1_confidence, L2_category, L1_category_name, L2_category_name

✅ Exported taxonomy definition to: cx_taxonomy_final.json

✅ Exported summary report to: cx_taxonomy_report.json

📊 FINAL TAXONOMY SUMMARY

Total Contacts Classified: 1,000
Classification Rate: 100.0%

🏷️  L1 Categories (6):

   L1_01: Order Status and Delivery Issues (25.3%)
   Covers inquiries related to order status, delivery delays, missing or incorrect deliveries, and related updates.
   L2 Subcategories (5):
      └─ Order Status Inquiries: 38.3% of Order Status and Delivery Issues
      └─ Delivery Delays: 23.7% of Order Status and Delivery Issues
      └─ Missing or Incorrect Deliveries: 37.1% of Order Status and Delivery Issues
      └─ Technical or App Issues: 0.4% of Order Stat

In [21]:
# Load taxonomy from JSON file
import json

with open('taxonomy_complete_proposal.json', 'r') as f:
    taxonomy = json.load(f)

print(f"✅ Loaded taxonomy with {len(taxonomy['L1_categories'])} L1 categories")

# ============================================
# CREATE SLIDE-READY CONTENT
# ============================================

print("="*80)
print("🎯 CX CONTACT REASON TAXONOMY")
print("="*80)
print(f"\n📊 Overview:")
print(f"   • {len(taxonomy['L1_categories'])} Top-Level (L1) Categories")
print(f"   • {sum(len(l1.get('L2_categories', [])) for l1 in taxonomy['L1_categories'])} Subcategories (L2)")
print(f"   • Built from analysis of 36,842 contact transcripts")
print(f"   • Stratified sampling: 70% phone/voice, 30% email")

print(f"\n📋 L1 Categories:\n")

for i, l1 in enumerate(taxonomy['L1_categories'], 1):
    print(f"{i}. {l1['name']} (~{l1.get('estimated_pct', 0)}%)")
    print(f"   {l1['description']}")
    print(f"   🔍 L2 Subcategories ({len(l1.get('L2_categories', []))}):")
    
    for l2 in l1.get('L2_categories', []):
        print(f"      • {l2['name']}")
    print()

# Create markdown for slides
with open('taxonomy_for_slides.md', 'w') as f:
    f.write("# CX Contact Reason Taxonomy\n\n")
    f.write("## Overview\n\n")
    f.write(f"- **{len(taxonomy['L1_categories'])}** Top-Level Categories\n")
    f.write(f"- **{sum(len(l1.get('L2_categories', [])) for l1 in taxonomy['L1_categories'])}** Subcategories\n")
    f.write(f"- Built from **36,842** contact transcripts\n\n")
    
    f.write("---\n\n")
    
    for l1 in taxonomy['L1_categories']:
        f.write(f"## {l1['name']}\n\n")
        f.write(f"**{l1['description']}**\n\n")
        f.write(f"### Subcategories:\n\n")
        for l2 in l1.get('L2_categories', []):
            f.write(f"- **{l2['name']}**: {l2['description']}\n")
        f.write(f"\n---\n\n")

print("💾 Saved markdown for slides to: taxonomy_for_slides.md")

✅ Loaded taxonomy with 6 L1 categories
🎯 CX CONTACT REASON TAXONOMY

📊 Overview:
   • 6 Top-Level (L1) Categories
   • 48 Subcategories (L2)
   • Built from analysis of 36,842 contact transcripts
   • Stratified sampling: 70% phone/voice, 30% email

📋 L1 Categories:

1. Order Status and Delivery Issues (~30%)
   Covers inquiries related to order status, delivery delays, missing or incorrect deliveries, and related updates.
   🔍 L2 Subcategories (8):
      • Order Status Inquiries
      • Delivery Delays
      • Missing or Incorrect Deliveries
      • Order Cancellations
      • Refunds and Compensation
      • Technical or App Issues
      • Shopper or Driver Behavior
      • Order Modifications

2. Order Modifications and Cancellations (~25%)
   Covers requests to modify or cancel orders due to customer changes, unavailable items, or system issues.
   🔍 L2 Subcategories (8):
      • Duplicate Order Cancellations
      • Unavailable Items Cancellations
      • Wrong Address Deliveries


In [22]:
iq.upload(df_final, "SANDBOX_DB_PII.SRIVIDYASEKAR.CX_RETAILER_CONTACTS_EVAL", if_exists="replace")

True

## gold dataset thats includes the topic summarization and information extraction 

## Contact Information Extraction System

This section implements a comprehensive system to extract structured information from customer service contacts (voice calls and emails).

### Key Improvements:
1. **Multi-Channel Support**: Handles both voice and email contacts with channel-specific guidance
2. **Context-Aware**: Uses subject line for additional context (especially for emails)
3. **Structured Extraction**: Extracts 7 fields including summary, contact reason, audience, retailer, product, order ID, and JIRA ticket
4. **Production-Ready**: Includes parallel processing, checkpointing, error handling, and progress tracking

### How to Use:
See the cells below for the prompt template, helper functions, and complete processing pipeline.


In [23]:
sql_query = """select * from SANDBOX_DB_PII.SRIVIDYASEKAR.CX_RETAILER_CONTACTS_SUMMARY"""
schema_df = iq.query(sql_query)

In [24]:
# Stratified sampling: 350 phone + 150 email = 500 total
# Filter for phone and email only first
schema_df_filtered = schema_df[schema_df["contact_channel"].isin(["phone", "email"])].copy()

# Sample 350 phone records
phone_sample = schema_df_filtered[schema_df_filtered["contact_channel"] == "phone"].sample(n=350, random_state=42)

# Sample 150 email records  
email_sample = schema_df_filtered[schema_df_filtered["contact_channel"] == "email"].sample(n=150, random_state=42)

# Combine the samples
schema_df = pd.concat([phone_sample, email_sample], ignore_index=True)

# Shuffle to mix phone and email records
schema_df = schema_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Verify the distribution
print(f"Total records: {len(schema_df)}")
print(f"\nChannel distribution:")
print(schema_df["contact_channel"].value_counts())
print(f"\nChannel percentages:")
print(schema_df["contact_channel"].value_counts() / len(schema_df))
schema_df.info()




Total records: 500

Channel distribution:
contact_channel
phone    350
email    150
Name: count, dtype: int64

Channel percentages:
contact_channel
phone    0.7
email    0.3
Name: count, dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   primary_contact_id              500 non-null    object
 1   contact_channel                 500 non-null    object
 2   transcript_created_date_at_utc  500 non-null    object
 3   is_retail_agent                 500 non-null    int8  
 4   subject                         148 non-null    object
 5   transcript                      500 non-null    object
 6   summary                         500 non-null    object
 7   error                           0 non-null      object
dtypes: int8(1), object(7)
memory usage: 28.0+ KB


In [25]:
SchemaSystemPrompt = """
You are a customer experience analyst specializing in contact reason classification. 

CONTEXT PROVIDED:
- Contact Channel: {CONTACT_CHANNEL} (call or email)
- Subject Line: {SUBJECT} (only for email)
- Transcript: {TRANSCRIPT}

YOUR TASK:
Analyze the contact and extract structured information. Consider the subject line for context, but prioritize the transcript content for classification.

EXTRACTION REQUIREMENTS:

1. summary (REQUIRED)
   - Write a single, concise sentence (≤ 25 words) capturing the PRIMARY reason for contact
   - Start with an action verb: "Retailer reported...", "Shopper requested...", "Customer inquired..."
   - Focus on the CORE ISSUE or REQUEST, not secondary concerns
   - Use neutral, descriptive language
   - Include key details: order/delivery issues, payment problems, product concerns, account issues, etc.
   - DO NOT include:
     * PII (names, emails, phone numbers)
     * Agent names or internal processes
     * Phrases like "the email thread" or "the caller"
     * Pleasantries or conversational filler
     * Resolution details (focus on the initial problem)
     * Multiple issues (choose the primary one)

2. contact_reason (REQUIRED)
   - Return EXACTLY one value from this list (copy exactly as written):
     *      Account and Membership Issues :: Account Linking and Information Updates
     * Account and Membership Issues :: Membership Management
     * Account and Membership Issues :: General Account Inquiries
     * Account and Membership Issues :: Account Access Issues
     * Account and Membership Issues :: Account Closure Requests
     * Account and Membership Issues :: Other
     * Order Cancellations and Modifications :: Retailer-Initiated Cancellations
     * Order Cancellations and Modifications :: Order Modification Requests
     * Order Cancellations and Modifications :: Delivery or Address Issues
     * Order Cancellations and Modifications :: Out-of-Stock or Substitution Issues
     * Order Cancellations and Modifications :: Customer-Initiated Cancellations
     * Order Cancellations and Modifications :: Wrong Order or Delivery Mix-Ups
     * Order Cancellations and Modifications :: Payment or Authorization Failures
     * Order Cancellations and Modifications :: System or Technical Issues
     * Order Cancellations and Modifications :: Other
     * Order Status and Delivery Issues :: Delivery Proof and Verification
     * Order Status and Delivery Issues :: Order Status Updates
     * Order Status and Delivery Issues :: Missing Deliveries
     * Order Status and Delivery Issues :: Delayed Deliveries
     * Order Status and Delivery Issues :: Address Updates and Delivery Instructions
     * Order Status and Delivery Issues :: Incorrect Delivery Location
     * Order Status and Delivery Issues :: Incorrect or Missing Items
     * Order Status and Delivery Issues :: Other
     * Payment and Refund Issues :: Declined Payments
     * Payment and Refund Issues :: Overcharges
     * Payment and Refund Issues :: Incorrect Charges for Promotions or Discounts
     * Payment and Refund Issues :: Refund Delays or Missing Refunds
     * Payment and Refund Issues :: Payment Processing Errors
     * Payment and Refund Issues :: Unauthorized or Fraudulent Charges
     * Payment and Refund Issues :: Other
     * Shopper Issues :: Shopper Misconduct
     * Shopper Issues :: Shopper Reassignment
     * Shopper Issues :: Shopper Performance Issues
     * Shopper Issues :: Other
     * Technical and App Issues :: Login and Access Issues
     * Technical and App Issues :: Order Processing Errors
     * Technical and App Issues :: Connectivity and Network Issues
     * Technical and App Issues :: Device Malfunctions
     * Technical and App Issues :: Order Cancellation and Modification Errors
     * Technical and App Issues :: App Performance Issues
     * Technical and App Issues :: Caper Carts Hardware Issues
     * Technical and App Issues :: Communication and Call Issues
     * Technical and App Issues :: Other
     * Store Closures/Hours :: Store Closures and Adjusted Hours
     * Store Closures/Hours :: Power and Network Outages
     * Store Closures/Hours :: Other
     * Other :: Other
   - If none fit well, return "Other :: Other"

3. contact_audience (REQUIRED)
   - Return EXACTLY one value (copy exactly as written):
     * Shopper
     * Internal Employee
     * Retailer / Retailer Account Manager
     * Customer
   - Determine based on who is contacting support, not who is being discussed

4. retailer (optional, can be null)
   - Return the retailer brand name as it appears in context (e.g., "Kroger", "Publix", "Costco", "Sprouts Farmers Market")
   - If multiple retailers appear, choose the one that is the primary subject of the issue
   - If no retailer is mentioned or identifiable, return null

5. product (optional, can be null)
   - Allowed values: "Storefront Pro", "Storefront", "Connect", "LMD", "IPP", "Caper"
   - Map variants/synonyms:
     * "SFP" / "storefront pro" → "Storefront Pro"
     * "last mile delivery" / "last-mile delivery" → "LMD"
   - Special cases:
     * If about Caper carts or smart cart technology → "Caper"
     * If about Instacart Platform Portal → "IPP"
   - Return comma-separated values if multiple products are mentioned (e.g., "Storefront Pro, Caper")
   - If no product is mentioned, return null

6. order_id (optional, can be null)
   - Extract the order ID if mentioned in the transcript
   - Return as a string of numbers only (e.g., "123456789")
   - If multiple order IDs appear, choose the one that is the primary subject of the issue
   - If no order ID is mentioned, return null

7. jira_ticket (optional, can be null)
   - Extract the JIRA ticket if mentioned (format: "PSO-[number]", e.g., "PSO-12345")
   - If multiple tickets appear, choose the one that is the primary subject of the issue
   - If no JIRA ticket is mentioned, return null

8. notes (optional, can be null)
   - Write any additional notes about the contact that are not covered by the other fields
   - Include any details that are interesting or important
   - Do not include PII, agent names, or internal process details

CHANNEL-SPECIFIC GUIDANCE:
- For VOICE contacts: Focus on the conversational flow; the subject is usually null
- For EMAIL contacts: Use the subject line to understand context, but rely on email body for classification

RETURN FORMAT:
Return ONLY valid JSON with exactly these keys (no extra text, no markdown, no code fences):
{{
  "summary": "",
  "contact_reason": "",
  "contact_audience": "",
  "retailer": null,
  "product": null,
  "order_id": null,
  "jira_ticket": null,
  "notes": null
}}
""".strip()

In [26]:
# Helper function to format the prompt with contact data
def format_extraction_prompt(transcript, subject="", contact_channel="email"):
    """
    Format the SchemaSystemPrompt with actual contact data.
    
    Parameters:
    - transcript (str): The full transcript/email body
    - subject (str): The subject line (use "" if not applicable, e.g., for voice calls)
    - contact_channel (str): Either "voice" or "email"
    
    Returns:
    - str: Formatted prompt ready for GPT
    """
    # Truncate long transcripts if needed (to manage token limits)
    max_transcript_length = 8000
    if len(transcript) > max_transcript_length:
        transcript = transcript[:max_transcript_length] + "\n\n[Transcript truncated for length...]"
    
    # Format the prompt
    formatted_prompt = SchemaSystemPrompt.format(
        CONTACT_CHANNEL=contact_channel,
        SUBJECT=subject if subject else "(No subject provided)",
        TRANSCRIPT=transcript
    )
    
    return formatted_prompt


# Example usage:
"""
# For email contacts:
prompt = format_extraction_prompt(
    transcript=row['transcript'],
    subject=row['subject'],
    contact_channel='email'
)

# For voice/phone contacts:
prompt = format_extraction_prompt(
    transcript=row['transcript'],
    subject='',  # Usually no subject for voice calls
    contact_channel='voice'
)

# Then call OpenAI:
response = client.chat.completions.create(
    model="gpt-4o-2024-11-20",
    temperature=0.1,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

# Parse the JSON response:
result = json.loads(response.choices[0].message.content)
"""


'\n# For email contacts:\nprompt = format_extraction_prompt(\n    transcript=row[\'transcript\'],\n    subject=row[\'subject\'],\n    contact_channel=\'email\'\n)\n\n# For voice/phone contacts:\nprompt = format_extraction_prompt(\n    transcript=row[\'transcript\'],\n    subject=\'\',  # Usually no subject for voice calls\n    contact_channel=\'voice\'\n)\n\n# Then call OpenAI:\nresponse = client.chat.completions.create(\n    model="gpt-4o-2024-11-20",\n    temperature=0.1,\n    messages=[\n        {"role": "user", "content": prompt}\n    ]\n)\n\n# Parse the JSON response:\nresult = json.loads(response.choices[0].message.content)\n'

In [27]:
# Complete processing function with parallel execution
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

def extract_contact_info(transcript, subject="", contact_channel="email"):
    """
    Extract structured information from a single contact using GPT.
    
    Returns:
    - dict: Extracted fields or error information
    """
    try:
        # Format the prompt
        prompt = format_extraction_prompt(transcript, subject, contact_channel)
        
        # Call OpenAI API
        response = client.chat.completions.create(
            model="gpt-4o-2024-11-20",
            temperature=0.1,
            max_tokens=300,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        # Parse JSON response
        result = json.loads(response.choices[0].message.content)
        
        return result
        
    except json.JSONDecodeError as e:
        return {
            "summary": "ERROR: Invalid JSON response",
            "contact_reason": "ERROR",
            "contact_audience": "ERROR",
            "retailer": None,
            "product": None,
            "order_id": None,
            "jira_ticket": None,
            "error_details": str(e),
            "notes": None
        }
    except Exception as e:
        return {
            "summary": f"ERROR: {str(e)}",
            "contact_reason": "ERROR",
            "contact_audience": "ERROR",
            "retailer": None,
            "product": None,
            "order_id": None,
            "jira_ticket": None,
            "error_details": str(e),
            "notes": None
        }


def process_contacts_parallel(df, max_workers=10, checkpoint_every=100, checkpoint_file='contact_extraction_checkpoint.csv'):
    """
    Process contacts in parallel with checkpointing.
    
    Parameters:
    - df: DataFrame with columns: 'transcript', 'subject' (optional), 'contact_channel'
    - max_workers: Number of parallel threads
    - checkpoint_every: Save checkpoint every N records
    - checkpoint_file: Path to checkpoint file
    
    Returns:
    - DataFrame with extracted fields added
    """
    
    # Check for checkpoint file
    if os.path.exists(checkpoint_file):
        print(f"📂 Found checkpoint file: {checkpoint_file}")
        print("   Loading previous progress...")
        checkpoint_df = pd.read_csv(checkpoint_file)
        print(f"   ✅ Loaded {len(checkpoint_df)} previously processed records")
        return checkpoint_df
    
    # Ensure required columns exist
    if 'subject' not in df.columns:
        df['subject'] = ""
    if 'contact_channel' not in df.columns:
        print("⚠️  Warning: 'contact_channel' column not found. Defaulting to 'email'")
        df['contact_channel'] = 'email'
    
    # Prepare results
    results = []
    
    # Helper function for parallel processing
    def process_row(idx, row):
        try:
            result = extract_contact_info(
                transcript=row['transcript'],
                subject=row.get('subject', ''),
                contact_channel=row.get('contact_channel', 'email')
            )
            return idx, result
        except Exception as e:
            return idx, {
                "summary": f"ERROR: {str(e)}",
                "contact_reason": "ERROR",
                "contact_audience": "ERROR",
                "retailer": None,
                "product": None,
                "order_id": None,
                "jira_ticket": None
            }
    
    # Process in parallel
    print(f"🚀 Starting parallel extraction with {max_workers} workers...")
    print(f"   Processing {len(df)} contacts")
    
    start_time = time.time()
    completed = 0
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {executor.submit(process_row, idx, row): idx for idx, row in df.iterrows()}
        
        # Process completed tasks with progress bar
        with tqdm(total=len(df), desc="Extracting") as pbar:
            for future in as_completed(futures):
                idx, result = future.result()
                results.append((idx, result))
                completed += 1
                pbar.update(1)
                
                # Checkpoint
                if completed % checkpoint_every == 0:
                    elapsed = time.time() - start_time
                    rate = completed / elapsed if elapsed > 0 else 0
                    eta = (len(df) - completed) / rate / 60 if rate > 0 else 0
                    
                    # Create checkpoint
                    checkpoint_data = []
                    for result_idx, result_data in results:
                        row_data = df.loc[result_idx].to_dict()
                        row_data.update(result_data)
                        checkpoint_data.append(row_data)
                    
                    checkpoint_df = pd.DataFrame(checkpoint_data)
                    checkpoint_df.to_csv(checkpoint_file, index=False)
                    
                    print(f"\n💾 Checkpoint saved. Progress: {completed}/{len(df)} ({100*completed/len(df):.1f}%)")
                    print(f"   Rate: {rate:.1f} rows/sec, ETA: {eta:.1f} min\n")
    
    # Create final DataFrame
    print("\n🎉 Processing complete! Building final DataFrame...")
    final_data = []
    for idx, result in results:
        row_data = df.loc[idx].to_dict()
        row_data.update(result)
        final_data.append(row_data)
    
    result_df = pd.DataFrame(final_data)
    
    # Save final checkpoint
    result_df.to_csv(checkpoint_file, index=False)
    print(f"✅ Final results saved to: {checkpoint_file}")
    
    return result_df




In [28]:
# Assuming you have a DataFrame 'df_contacts' with columns:
# - transcript (text)
# - subject (text, optional)
# - contact_channel ('voice' or 'email')

df_extracted = process_contacts_parallel(
    schema_df,
    max_workers=10,
    checkpoint_every=100,
    checkpoint_file='contact_extraction_results.csv'
)

# Check results
print(df_extracted[['summary', 'contact_reason', 'contact_audience', 'retailer','product','order_id','jira_ticket','notes']].head())

# Count distribution
print("\nContact Reason Distribution:")
print(df_extracted['contact_reason'].value_counts())

print("\nContact Audience Distribution:")
print(df_extracted['contact_audience'].value_counts())

🚀 Starting parallel extraction with 10 workers...
   Processing 500 contacts


Extracting:  20%|██        | 101/500 [00:12<00:55,  7.23it/s]


💾 Checkpoint saved. Progress: 100/500 (20.0%)
   Rate: 7.8 rows/sec, ETA: 0.9 min



Extracting:  40%|████      | 202/500 [00:23<00:35,  8.40it/s]


💾 Checkpoint saved. Progress: 200/500 (40.0%)
   Rate: 8.7 rows/sec, ETA: 0.6 min



...


💾 Checkpoint saved. Progress: 300/500 (60.0%)
   Rate: 8.8 rows/sec, ETA: 0.4 min



Extracting:  80%|███████▉  | 398/500 [00:43<00:08, 11.51it/s]


💾 Checkpoint saved. Progress: 400/500 (80.0%)
   Rate: 9.1 rows/sec, ETA: 0.2 min



Extracting: 100%|██████████| 500/500 [00:55<00:00,  9.06it/s]



💾 Checkpoint saved. Progress: 500/500 (100.0%)
   Rate: 9.1 rows/sec, ETA: 0.0 min


🎉 Processing complete! Building final DataFrame...
✅ Final results saved to: contact_extraction_results.csv
                                             summary  \
0  Retailer reported a declined payment due to a ...   
1  Retailer reported early store closure due to s...   
2  Retailer requested to cancel an order placed a...   
3  Retailer reported a driver arriving at the wro...   
4  Retailer reported an order stuck in shopping a...   

                                      contact_reason  \
0     Payment and Refund Issues :: Declined Payments   
1  Store Closures/Hours :: Store Closures and Adj...   
2  Order Cancellations and Modifications :: Custo...   
3  Order Status and Delivery Issues :: Incorrect ...   
4  Order Status and Delivery Issues :: Order Stat...   

                      contact_audience                retailer product  \
0  Retailer / Retailer Account Manager                  Wa

In [29]:
# Define regex patterns and masking functions for PII
import re

# Regex patterns for email and phone
EMAIL_RE = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')
PHONE_RE = re.compile(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b')

# Masking functions
def mask_email(match):
    """Mask email addresses"""
    email = match.group(0)
    parts = email.split('@')
    if len(parts) == 2:
        return f"[EMAIL-{parts[1]}]"
    return "[EMAIL]"

def mask_phone(match):
    """Mask phone numbers"""
    return "[PHONE]"

def mask_names(text, names=None, spacy_model=None):
    """
    Mask person names in text.
    
    Args:
        text: Input text to mask
        names: Optional list of specific names to mask
        spacy_model: Optional spaCy model for NER-based name detection
    
    Returns:
        Text with names masked as [NAME]
    """
    if not text:
        return text
    
    masked_text = text
    
    # Mask specific names if provided
    if names:
        for name in names:
            masked_text = re.sub(
                rf'\b{re.escape(name)}\b', 
                '[NAME]', 
                masked_text, 
                flags=re.IGNORECASE
            )
    
    # Use spaCy for NER-based name detection if model provided
    if spacy_model:
        try:
            import spacy
            nlp = spacy.load(spacy_model)
            doc = nlp(masked_text)
            # Replace PERSON entities with [NAME]
            for ent in reversed(doc.ents):
                if ent.label_ == 'PERSON':
                    masked_text = masked_text[:ent.start_char] + '[NAME]' + masked_text[ent.end_char:]
        except Exception as e:
            print(f"Warning: Could not use spaCy for name masking: {e}")
    
    return masked_text


def mask_pii_extended(text, mask_names_list=None, spacy_model=None):
    """
    Complete PII masking pipeline that applies all masking functions.
    
    Args:
        text: Input text to mask
        mask_names_list: Optional list of specific names to mask
        spacy_model: Optional spaCy model for NER-based name detection
    
    Returns:
        Text with all PII masked (emails, phones, names)
    """
    if text is None:
        return text
    
    # Convert to string
    masked = str(text)
    
    # Mask emails
    masked = EMAIL_RE.sub(mask_email, masked)
    
    # Mask phone numbers
    masked = PHONE_RE.sub(mask_phone, masked)
    
    # Mask names
    masked = mask_names(masked, names=mask_names_list, spacy_model=spacy_model)
    
    return masked


print("PII masking functions loaded!")
print("Available functions:")
print("  - mask_email(): Mask email addresses")
print("  - mask_phone(): Mask phone numbers")
print("  - mask_names(): Mask person names")
print("  - mask_pii_extended(): Complete PII masking pipeline (recommended)")


PII masking functions loaded!
Available functions:
  - mask_email(): Mask email addresses
  - mask_phone(): Mask phone numbers
  - mask_names(): Mask person names
  - mask_pii_extended(): Complete PII masking pipeline (recommended)


In [30]:
# Apply extended masking to both transcript and transcript_cleaned
custom_names = []  # optionally add known names to mask
spacy_model = None  # optionally set to a spaCy model name

df_extracted = df_extracted.copy()

print("Applying PII masking to transcript columns...")

# Mask original transcript column
if 'transcript' in df_extracted.columns:
    print("  - Masking 'transcript' → 'transcript_masked'")
    df_extracted['transcript_masked'] = df_extracted['transcript'].astype(str).apply(
        lambda t: mask_pii_extended(t, mask_names_list=custom_names, spacy_model=spacy_model)
    )
    print(f"    ✓ Completed {len(df_extracted)} records")

# Mask cleaned transcript column (if it exists)
if 'transcript_cleaned' in df_extracted.columns:
    print("  - Masking 'transcript_cleaned' → 'transcript_cleaned_masked'")
    df_extracted['transcript_cleaned_masked'] = df_extracted['transcript_cleaned'].astype(str).apply(
        lambda t: mask_pii_extended(t, mask_names_list=custom_names, spacy_model=spacy_model)
    )
    print(f"    ✓ Completed {len(df_extracted)} records")
else:
    print("  - 'transcript_cleaned' column not found, skipping")

print(f"\n✅ PII masking complete!")
print(f"Columns in df_extracted: {list(df_extracted.columns)}")

Applying PII masking to transcript columns...
  - Masking 'transcript' → 'transcript_masked'
    ✓ Completed 500 records
  - 'transcript_cleaned' column not found, skipping

✅ PII masking complete!
Columns in df_extracted: ['primary_contact_id', 'contact_channel', 'transcript_created_date_at_utc', 'is_retail_agent', 'subject', 'transcript', 'summary', 'error', 'contact_reason', 'contact_audience', 'retailer', 'product', 'order_id', 'jira_ticket', 'notes', 'transcript_masked']


In [31]:
import pandas as pd
import re


# Find one email and one phone example
email_example = df_extracted[df_extracted['contact_channel'] == 'email'].iloc[0] if len(df_extracted[df_extracted['contact_channel'] == 'email']) > 0 else None
phone_example = df_extracted[df_extracted['contact_channel'] == 'phone'].iloc[0] if len(df_extracted[df_extracted['contact_channel'] == 'phone']) > 0 else None

# Enhanced PII masking function
def mask_pii(text):
    if pd.isna(text) or text == "":
        return text
    
    masked_text = str(text)
    
    # 1. SPELLED-OUT PHONE NUMBERS (for voice transcripts)
    number_words = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
        'oh': '0'
    }
    
    word_pattern = r'\b(?:' + '|'.join(number_words.keys()) + r')\b'
    matches = list(re.finditer(word_pattern, masked_text, re.IGNORECASE))
    
    i = 0
    while i < len(matches):
        sequence_start = i
        sequence_end = i
        
        while sequence_end < len(matches) - 1:
            current_end = matches[sequence_end].end()
            next_start = matches[sequence_end + 1].start()
            if next_start - current_end < 50:
                sequence_end += 1
            else:
                break
        
        sequence_length = sequence_end - sequence_start + 1
        
        if sequence_length >= 7:
            start_pos = matches[sequence_start].start()
            end_pos = matches[sequence_end].end()
            masked_text = masked_text[:start_pos] + '[PHONE_SPOKEN]' + masked_text[end_pos:]
            matches = list(re.finditer(word_pattern, masked_text, re.IGNORECASE))
            i = 0
        else:
            i = sequence_end + 1
    
    # 2. Names in conversational context
    masked_text = re.sub(
        r'(?:my name is|i am|i\'m|this is|named)\s+([a-z]+)',
        r'\1 [NAME]',
        masked_text,
        flags=re.IGNORECASE
    )
    
    # 3. Email addresses
    masked_text = re.sub(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        '[EMAIL]',
        masked_text
    )
    
    # 4. Phone numbers (numeric)
    masked_text = re.sub(
        r'\b(?:\+?1[-.]?)?\(?([0-9]{3})\)?[-.]?([0-9]{3})[-.]?([0-9]{4})\b',
        '[PHONE]',
        masked_text
    )
    
    # 5. Credit cards
    masked_text = re.sub(
        r'\b(?:\d[ -]*?){13,19}\b',
        '[CREDIT_CARD]',
        masked_text
    )
    
    # 6. Order IDs
    masked_text = re.sub(
        r'\b(?:order|account|id|number|#)[:\s]*([0-9]{6,})\b',
        r'[ORDER_ID]',
        masked_text,
        flags=re.IGNORECASE
    )
    
    # 7. URLs
    masked_text = re.sub(
        r'https?://[^\s]+',
        '[URL]',
        masked_text
    )
    
    return masked_text

# Print email example
if email_example is not None:
    print("="*80)
    print("📧 EMAIL EXAMPLE")
    print("="*80)
    print("\n--- ORIGINAL SUBJECT ---")
    print(email_example['subject'][:200] if pd.notna(email_example['subject']) else "(No subject)")
    print("\n--- MASKED SUBJECT ---")
    print(mask_pii(email_example['subject'])[:200] if pd.notna(email_example['subject']) else "(No subject)")
    print("\n--- ORIGINAL TRANSCRIPT (first 500 chars) ---")
    print(email_example['transcript'][:500])
    print("\n--- MASKED TRANSCRIPT (first 500 chars) ---")
    print(mask_pii(email_example['transcript'])[:500])
    print("\n")

# Print phone example
if phone_example is not None:
    print("="*80)
    print("📱 PHONE/VOICE EXAMPLE")
    print("="*80)
    print("\n--- ORIGINAL SUBJECT ---")
    print(phone_example['subject'][:200] if pd.notna(phone_example['subject']) else "(No subject)")
    print("\n--- MASKED SUBJECT ---")
    print(mask_pii(phone_example['subject'])[:200] if pd.notna(phone_example['subject']) else "(No subject)")
    print("\n--- ORIGINAL TRANSCRIPT (first 600 chars) ---")
    print(phone_example['transcript'][:600])
    print("\n--- MASKED TRANSCRIPT (first 600 chars) ---")
    print(mask_pii(phone_example['transcript'])[:600])
    print("\n")


📧 EMAIL EXAMPLE

--- ORIGINAL SUBJECT ---
[EXTERNAL] Store Hours Change Tracker: Store#865 Hours Change

--- MASKED SUBJECT ---
[EXTERNAL] Store Hours Change Tracker: Store#865 Hours Change

--- ORIGINAL TRANSCRIPT (first 500 chars) ---
Message 1 (2025-07-12 19:49:42.000 Z):
Store#: 865
Store Name: Alpharetta
Address: 7300 North Point Pkwy, Apharetta, GA-30022
DSM/MSM#: John Williams
Reason: EARLY CLOSING
Special Hours: SAT 9:30-6
Special Hours Effective Date: 07/12/2025
Notes: Early closing due to staffing challenges


To unsubscribe from this group and stop receiving emails from it, send an email to StoreHoursAdjustments+unsubscribe@instacart.com.

--- MASKED TRANSCRIPT (first 500 chars) ---
Message 1 (2025-07-12 19:49:42.000 Z):
Store#: 865
Store Name: Alpharetta
Address: 7300 North Point Pkwy, Apharetta, GA-30022
DSM/MSM#: John Williams
Reason: EARLY CLOSING
Special Hours: SAT 9:30-6
Special Hours Effective Date: 07/12/2025
Notes: Early closing due to staffing challenges


To unsub

In [32]:
iq.upload(df_extracted, "SANDBOX_DB_PII.SRIVIDYASEKAR.CX_RETAILER_CONTACTS_SCHEMA", if_exists="replace")

True

In [33]:
df_extracted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   primary_contact_id              500 non-null    object
 1   contact_channel                 500 non-null    object
 2   transcript_created_date_at_utc  500 non-null    object
 3   is_retail_agent                 500 non-null    int64 
 4   subject                         148 non-null    object
 5   transcript                      500 non-null    object
 6   summary                         500 non-null    object
 7   error                           0 non-null      object
 8   contact_reason                  500 non-null    object
 9   contact_audience                500 non-null    object
 10  retailer                        302 non-null    object
 11  product                         69 non-null     object
 12  order_id                        160 non-null    ob